# Optimization Performance vs. Variables Analysis

This notebook analyzes tracking performance across different variables:
1. Number of photon rays
2. Number of sensors
3. Particle energy

Refactored for better code organization and reusability.

In [ ]:
import pickle
import sys
sys.path.append('..')

import jax
import jax.numpy as jnp
import numpy as np

# Import matplotlib FIRST, before pyplot
import matplotlib
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.size'] = 12

# NOW import pyplot - it will use the settings above
from matplotlib import pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from scipy.optimize import curve_fit
import torch

from tools.geometry import generate_detector

In [ ]:
# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def extract_histories(all_event_results):
    """
    Extract convergence histories for all events.
    
    Returns:
        Dictionary with arrays for each metric, shape (n_events, n_iterations)
    """
    histories = {
        'position_errors': [],
        'direction_errors': [],
        't0_errors': [],
        'energy_errors': [],
        'combined_losses': [],
        'vertex_losses': [],
        'counts_losses': [],
        'energy_losses': []
    }
    
    for event_result in all_event_results:
        opt_results = event_result['optimization_results']
        history = opt_results['history']
        
        for key in histories.keys():
            histories[key].append(history[key])
    
    # Convert to numpy arrays
    for key in histories:
        histories[key] = np.array(histories[key])
    
    return histories


def compute_statistics(data_array):
    """
    Compute mean, median, 68th percentile, and 90th percentile.
    
    Args:
        data_array: numpy array of shape (n_events, n_iterations)
    
    Returns:
        Dictionary with statistical measures
    """
    return {
        'mean': np.mean(data_array, axis=0),
        'median': np.median(data_array, axis=0),
        'percentile_68': np.percentile(data_array, 68, axis=0),
        'percentile_90': np.percentile(data_array, 90, axis=0)
    }


def bootstrap_percentile_ci(data, percentile=95, n_bootstrap=1000, ci_level=68):
    """
    Compute bootstrap confidence intervals for a given percentile.
    """
    bootstrap_estimates = []
    n = len(data)
    for _ in range(n_bootstrap):
        bootstrap_sample = np.random.choice(data, size=n, replace=True)
        bootstrap_estimates.append(np.percentile(bootstrap_sample, percentile))
    bootstrap_estimates = np.array(bootstrap_estimates)
    alpha = (100 - ci_level) / 2
    ci_lower = np.percentile(bootstrap_estimates, alpha)
    ci_upper = np.percentile(bootstrap_estimates, 100 - alpha)
    return {
        "estimate": np.percentile(data, percentile),
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "std_error": np.std(bootstrap_estimates),
    }


def exp_like(x, a, b, c):
    """Exponential-like fit function: y = a * b^x + c"""
    return a * (b ** x) + c


def constant(x, q):
    """Constant fit function: y = q"""
    return np.full_like(x, q)


def compute_momentum_errors(energy_errors, true_energy, m_mu=105.658):
    """
    Convert energy errors to momentum errors (%).
    
    Args:
        energy_errors: Array of energy errors
        true_energy: True particle energy (MeV or GeV, must match m_mu units)
        m_mu: Muon mass (default 105.658 MeV, adjust if using GeV)
    
    Returns:
        Momentum errors as percentage
    """
    E_total = true_energy + m_mu
    p_mu = np.sqrt(E_total**2 - m_mu**2)
    conversion_factor = E_total / p_mu
    momentum_errors_percent = (conversion_factor * (energy_errors / E_total)) * 100
    return momentum_errors_percent

In [ ]:
# ============================================================
# DATA LOADING AND PROCESSING FUNCTIONS
# ============================================================

def load_and_process_results(results_files, variable_extractor, muon_mass=105.658):
    """
    Load and process results from multiple pickle files.
    
    Args:
        results_files: List of paths to pickle files
        variable_extractor: Function that extracts the x-variable from results_summary
        muon_mass: Muon mass in MeV or GeV (must match energy units)
    
    Returns:
        tuple: (x_values, histories_list, stats_list, results_summaries)
    """
    x_values = []
    histories_list = []
    stats_list = []
    results_summaries = []
    
    for results_file in results_files:
        # Load results
        with open(results_file, 'rb') as f:
            results_summary = pickle.load(f)
            results_summaries.append(results_summary)
        
        print(f"Loaded: {results_file}")
        
        # Extract x-variable value
        x_value = variable_extractor(results_summary)
        x_values.append(x_value)
        print(f"  X-variable value: {x_value}")
        
        # Extract histories
        all_event_results = results_summary['all_event_results']
        histories = extract_histories(all_event_results)
        
        # Compute statistics for all metrics
        stats = {key: compute_statistics(histories[key]) for key in histories}
        
        # Convert energy errors to momentum errors
        true_energy = x_value if 'energy' in results_file else results_summary['config'].get('true_energy', 1050.0)
        momentum_errors_percent = compute_momentum_errors(histories['energy_errors'], true_energy, muon_mass)
        histories['momentum_errors_percent'] = momentum_errors_percent
        stats['momentum_errors_percent'] = compute_statistics(momentum_errors_percent)
        
        histories_list.append(histories)
        stats_list.append(stats)
    
    return x_values, histories_list, stats_list, results_summaries


def extract_timing_metrics(results_summaries):
    """
    Extract timing metrics from results summaries.
    
    Returns:
        tuple: (total_times_mean, total_times_std, adam_times_mean, adam_times_std)
    """
    total_times_mean, total_times_std = [], []
    adam_times_mean, adam_times_std = [], []
    
    for results_summary in results_summaries:
        all_event_results = results_summary["all_event_results"]
        total_t = np.array([ev["total_event_time"] for ev in all_event_results])
        adam_t = np.array([ev.get("adam_optimization_time", np.nan) for ev in all_event_results])
        total_times_mean.append(np.nanmean(total_t))
        total_times_std.append(np.nanstd(total_t))
        adam_times_mean.append(np.nanmean(adam_t))
        adam_times_std.append(np.nanstd(adam_t))
    
    return total_times_mean, total_times_std, adam_times_mean, adam_times_std


def compute_bootstrap_metrics(histories_list, metrics, n_iter=-1, percentile=68, ci_level=68):
    """
    Compute bootstrap metrics for all histories.
    
    Args:
        histories_list: List of history dictionaries
        metrics: Dictionary defining metrics to compute {key: {"label": str, "scale": float}}
        n_iter: Iteration index to analyze (-1 for final)
        percentile: Percentile to compute
        ci_level: Confidence interval level
    
    Returns:
        Dictionary with bootstrap results for each metric
    """
    np.random.seed(42)
    metric_results = {key: {"y": [], "yerr": []} for key in metrics}
    
    valid_histories = [h for h in histories_list if isinstance(h.get("position_errors", None), np.ndarray)]
    
    for histories in valid_histories:
        for key, meta in metrics.items():
            data = histories[key][:, n_iter] * meta["scale"]
            boot = bootstrap_percentile_ci(data, percentile=percentile, ci_level=ci_level)
            metric_results[key]["y"].append(boot["estimate"])
            yerr = 0.5 * ((boot["ci_upper"] - boot["estimate"]) + (boot["estimate"] - boot["ci_lower"]))
            metric_results[key]["yerr"].append(yerr)
    
    return metric_results

In [ ]:
# ============================================================
# PLOTTING FUNCTIONS
# ============================================================

def plot_metrics_vs_variable(
    x_values,
    metric_results,
    metrics,
    xlabel,
    output_filename,
    x_scale=1.0,
    timing_data=None,
    ylims=None,
    fit_types=None,
    line_color="navy"
):
    """
    Create a multi-panel plot showing metrics vs. a variable.
    
    Args:
        x_values: Array of x-axis values
        metric_results: Dictionary of bootstrap results for each metric
        metrics: Dictionary defining metrics {key: {"label": str, "scale": float}}
        xlabel: Label for x-axis
        output_filename: Path to save figure
        x_scale: Scaling factor for x-values
        timing_data: Optional tuple (total_mean, total_std, adam_mean, adam_std)
        ylims: Optional dict of y-axis limits {metric_key: (min, max)}
        fit_types: Optional dict of fit types {metric_key: 'exp' or 'const'}
        line_color: Color for data points
    """
    n_panels = len(metrics) + (1 if timing_data else 0)
    fig, axes = plt.subplots(n_panels, 1, figsize=(6, 2.5 * n_panels), sharex=True)
    if n_panels == 1:
        axes = [axes]
    plt.subplots_adjust(hspace=0.05)
    
    xdata = np.array(x_values) / x_scale
    
    # Plot metrics
    for ax, (key, meta) in zip(axes[:len(metrics)], metrics.items()):
        y = np.array(metric_results[key]["y"])
        yerr = np.array(metric_results[key]["yerr"])
        
        ax.errorbar(
            xdata, y, yerr=yerr, fmt="o", capsize=4, color=line_color, label=meta["label"]
        )
        
        # Determine fit type
        fit_type = 'exp' if fit_types is None else fit_types.get(key, 'exp')
        
        # Fit curve
        try:
            if fit_type == 'const':
                popt, _ = curve_fit(
                    constant, xdata, y, sigma=yerr, absolute_sigma=True,
                    p0=[np.mean(y)], maxfev=10000
                )
                xfit = np.linspace(xdata.min(), xdata.max(), 200)
                yfit = constant(xfit, *popt)
                ax.plot(xfit, yfit, "-", color="cornflowerblue", lw=2)
                ax.text(
                    0.55, 0.9, f"$y = {popt[0]:.2f}$",
                    transform=ax.transAxes, fontsize=10, color="cornflowerblue",
                    ha="left", va="top"
                )
            else:
                popt, _ = curve_fit(
                    exp_like, xdata, y, sigma=yerr, absolute_sigma=True,
                    p0=[y.max() - y.min(), 0.99, y.min()], maxfev=10000
                )
                xfit = np.linspace(xdata.min(), xdata.max(), 200)
                yfit = exp_like(xfit, *popt)
                ax.plot(xfit, yfit, "-", color="cornflowerblue", lw=2)
                a, b, c = popt
                ax.text(
                    0.55, 0.9,
                    f"$y = {a:.2f}\\times\\,{b:.3f}^x + {c:.2f}$",
                    transform=ax.transAxes, fontsize=10, color="cornflowerblue",
                    ha="left", va="top"
                )
        except RuntimeError:
            print(f"⚠️ Fit failed for {key}")
        
        # Set y-axis limits if provided
        if ylims and key in ylims:
            ax.set_ylim(ylims[key])
        
        ax.set_ylabel(meta["label"])
        ax.grid(True, alpha=0.3)
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    
    # Plot timing data if provided
    if timing_data:
        total_mean, total_std, adam_mean, adam_std = timing_data
        ax_time = axes[-1]
        ax_time.errorbar(
            xdata, total_mean, yerr=total_std,
            fmt="o-", capsize=4, color="tab:red", label="Total"
        )
        ax_time.errorbar(
            xdata, adam_mean, yerr=adam_std,
            fmt="s--", capsize=4, color="tab:orange", label="Adam"
        )
        ax_time.set_ylabel("Time per event (s)")
        ax_time.grid(True, alpha=0.3)
        ax_time.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        ax_time.legend(frameon=False, loc="lower right")
        ax_time.set_ylim(0)
        ax_time.set_xlim(0)
    
    axes[-1].set_xlabel(xlabel)
    fig.align_ylabels(axes)
    plt.savefig(output_filename, bbox_inches='tight')
    plt.show()
    print(f"\nSaved figure to: {output_filename}")

## Analysis 1: Performance vs. Number of Rays

In [ ]:
# Define file paths and variable extractor
common_path = '/sdf/data/neutrino/cjesus/lucid_output/paper_files/'
results_files_nrays = [
    common_path + 'speed_results_5k_nrays.pkl',
    common_path + 'speed_results_10k_nrays.pkl',
    common_path + 'speed_results_25k_nrays.pkl',
    common_path + 'speed_results_50k_nrays.pkl',
    common_path + 'speed_results_100k_nrays.pkl',
    common_path + 'speed_results_150k_nrays.pkl'
]

# Extract nphot as x-variable
nphot_extractor = lambda rs: rs['config']['nphot']

# Load and process data
_, _, _, summaries_nrays = load_and_process_results(
    results_files_nrays, nphot_extractor, muon_mass=0.105658  # GeV
)

# Extract timing metrics
timing_nrays = extract_timing_metrics(summaries_nrays)

results_files_nrays = [
    common_path + 'results_5k_nrays.pkl',
    common_path + 'results_10k_nrays.pkl',
    common_path + 'results_25k_nrays.pkl',
    common_path + 'results_50k_nrays.pkl',
    common_path + 'results_100k_nrays.pkl',
    common_path + 'results_150k_nrays.pkl'
]

# Load and process data
x_nrays, hist_nrays, stats_nrays, summaries_nrays = load_and_process_results(
    results_files_nrays, nphot_extractor, muon_mass=0.105658  # GeV
)


# Define metrics
metrics = {
    "position_errors": {"label": "Pos. error (cm)", "scale": 100},
    "direction_errors": {"label": "Dir. error (°)", "scale": 1},
    "t0_errors": {"label": "t₀ error (ns)", "scale": 1},
    "momentum_errors_percent": {"label": "Mom. error (%)", "scale": 1},
}

# Compute bootstrap metrics
metric_results_nrays = compute_bootstrap_metrics(hist_nrays, metrics, ci_level=90)

# Plot
plot_metrics_vs_variable(
    x_nrays,
    metric_results_nrays,
    metrics,
    xlabel="Number of Rays ($\\times10^3$)",
    output_filename='figures/tracking_performance_vs_nrays.pdf',
    x_scale=1000.0,
    timing_data=timing_nrays,
    ylims={
        "position_errors": (10, 50),
        "direction_errors": (0.7, 2.3),
        "t0_errors": (0, 2.1),
        "momentum_errors_percent": (0, 16)
    }
)

## Analysis 2: Performance vs. Number of Sensors

In [ ]:
# Define file paths and variable extractor
results_files_sensors = [
    common_path + f'results_{i}k_geom_opt.pkl' 
    for i in range(2, 20)
]

# Extract number of sensors as x-variable
def sensors_extractor(rs):
    detector_config_filename = rs['config']['detector_file']
    detector = generate_detector(detector_config_filename)
    return len(detector.all_points)

# Load and process data
x_sensors, hist_sensors, stats_sensors, summaries_sensors = load_and_process_results(
    results_files_sensors, sensors_extractor, muon_mass=105.658  # MeV
)

# Compute bootstrap metrics
metric_results_sensors = compute_bootstrap_metrics(hist_sensors, metrics, ci_level=68)

# Plot (no timing panel for this analysis)
plot_metrics_vs_variable(
    x_sensors,
    metric_results_sensors,
    metrics,
    xlabel="Number of Sensors ($\\times10^3$)",
    output_filename='figures/detector_perf_vs_num_sensors.pdf',
    x_scale=1000.0,
    ylims={
        "position_errors": (10, 40),
        "direction_errors": (0.4, 1.8),
        "t0_errors": (0, 1.2),
        "momentum_errors_percent": (0, 4.5)
    }
)

## Analysis 3: Performance vs. Particle Energy

In [ ]:
# Define file paths and variable extractor
results_files_energy = [
    common_path + f'energy_{i}.pkl' 
    for i in range(2, 18)
]

# Extract energy as x-variable
def energy_extractor(rs):
    all_event_results = rs['all_event_results']
    true_energies = np.array([e['event_data']['true_energy'] for e in all_event_results])
    return int(np.unique(true_energies)[0])

# Load and process data
x_energy, hist_energy, stats_energy, summaries_energy = load_and_process_results(
    results_files_energy, energy_extractor, muon_mass=105.658  # MeV
)

# Compute bootstrap metrics
metric_results_energy = compute_bootstrap_metrics(hist_energy, metrics, ci_level=68)

# Plot with constant fit for momentum
plot_metrics_vs_variable(
    x_energy,
    metric_results_energy,
    metrics,
    xlabel="Energy (MeV)",
    output_filename='figures/tracking_performance_vs_energy.pdf',
    x_scale=1.0,
    fit_types={
        "position_errors": "exp",
        "direction_errors": "exp",
        "t0_errors": "exp",
        "momentum_errors_percent": "const"
    },
    ylims={
        "position_errors": (10, 35),
        "direction_errors": (0., 2.9),
        "t0_errors": (0, 1.6),
        "momentum_errors_percent": (0, 4.4)
    }
)